# Caso de Estudio: Logística y Telemetría GPS en Baja California
**Materia:** Minería de Datos  
**Objetivo:** Resolver dos problemas críticos de negocio (ubicación estratégica de Centros de Distribución y filtrado operativo de zonas urbanas vs. ruido rural) utilizando algoritmos de clusterización de `scikit-learn` (`KMeans` y `DBSCAN`).

---

## Fase 1: Exploración Inicial
1. Importamos el archivo CSV `data/logistica_baja_california.csv` utilizando `pandas`.
2. Aplicamos `StandardScaler` sobre las columnas `Longitud` y `Latitud` y guardamos la matriz escalada (`X_scaled`) para nivelar el terreno matemático entre grados de latitud y longitud.
3. Graficamos los datos crudos en un *Scatter Plot* (`Longitud` en el eje X, `Latitud` en el eje Y) para observar la silueta de la costa del Pacífico de Baja California.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN

# 1. Importar el CSV usando pandas
csv_path = os.path.join("data", "logistica_baja_california.csv")
df = pd.read_csv(csv_path)
display(df.head())
print(f"Total de registros cargados: {len(df)}")

# 2. Escalar Latitud y Longitud con StandardScaler
scaler = StandardScaler()
X_raw = df[['Longitud', 'Latitud']]
X_scaled = scaler.fit_transform(X_raw)

# 3. Graficar los datos crudos (Longitud en X, Latitud en Y)
plt.figure(figsize=(8, 9))
plt.scatter(df['Longitud'], df['Latitud'], s=15, alpha=0.6, color='#1f77b4', edgecolors='none')
plt.title('Fase 1: Telemetría GPS de Solicitudes en Baja California (Datos Crudos)', fontsize=13, fontweight='bold')
plt.xlabel('Longitud (Eje X)')
plt.ylabel('Latitud (Eje Y)')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

---

## Fase 2: Misión Estratégica (Ubicación de 3 Cedis)
### Selección del Algoritmo: **K-Means (`KMeans`)**
* **¿Por qué se adecua a este problema?** El problema pide ubicar exactamente **3 Centros de Distribución (`k = 3`)** en el **centro geográfico (centroide)** de la demanda general para minimizar la distancia de viaje hacia cualquier punto. Matemáticamente, **K-Means** particiona el conjunto de datos en $K$ clústeres definidos a priori y calcula explícitamente los centroides (`cluster_centers_`) que minimizan la suma de distancias euclidianas al cuadrado (inercia o *WCSS*) desde cada solicitud hasta su centro más cercano.

In [ ]:
# Entrenamos K-Means con 3 clústeres sobre la matriz escalada
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster_KMeans'] = kmeans.fit_predict(X_scaled)

# Extraemos los cluster_centers_ (en escala estándar) y también los des-escalamos a coordenadas GPS reales
centros_escalados = kmeans.cluster_centers_
centros_gps = scaler.inverse_transform(centros_escalados)

df_centros = pd.DataFrame(centros_gps, columns=['Longitud', 'Latitud'])
df_centros['Cedis'] = [f'Cedis {i}' for i in range(3)]
display(df_centros)

# Gráfico de dispersión pintando los 3 clústeres y los cluster_centers_ como "X" gigantes
plt.figure(figsize=(9, 9))
scatter = plt.scatter(
    df['Longitud'],
    df['Latitud'],
    c=df['Cluster_KMeans'],
    cmap='viridis',
    s=20,
    alpha=0.7,
    label='Solicitudes de servicio'
)

plt.scatter(
    centros_gps[:, 0],
    centros_gps[:, 1],
    c='red',
    marker='X',
    s=350,
    edgecolors='black',
    linewidths=1.5,
    label='Centroides (Cedis)'
)

for i, row in df_centros.iterrows():
    plt.annotate(
        f"Cedis {i}\n({row['Latitud']:.2f}, {row['Longitud']:.2f})",
        (row['Longitud'], row['Latitud']),
        textcoords="offset points",
        xytext=(12, -5),
        fontsize=9,
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
    )

plt.title('Fase 2: Misión Estratégica - Ubicación de 3 Cedis con K-Means', fontsize=13, fontweight='bold')
plt.xlabel('Longitud (Eje X)')
plt.ylabel('Latitud (Eje Y)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

### Preguntas de Análisis — Fase 2

1. **¿Dónde colocó el algoritmo que seleccionaron los 3 Cedis? ¿Consideran que sus localizaciones hacen sentido?**
   * **Cedis Norte (`Lat: ~32.51, Lon: ~-117.04`):** Se ubicó directamente sobre la mancha urbana de **Tijuana** (ligeramente desplazado hacia el sur por los puntos de Rosarito). Esta ubicación **sí hace sentido** por el enorme volumen de pedidos locales.
   * **Cedis Centro (`Lat: ~31.91, Lon: ~-116.58`):** Quedó posicionado en un punto intermedio en carretera **entre Ensenada y el Valle de Guadalupe**, debido a que el algoritmo fusionó ambas manchas urbanas dentro de un mismo clúster.
   * **Cedis Sur (`Lat: ~31.04, Lon: ~-116.14`):** Quedó ubicado **en medio de la zona rural/carretera al sur del estado** (a la altura de Colonet / Camalú / San Telmo), atendiendo a menos de 100 solicitudes aisladas a lo largo de la carretera Transpeninsular.
   * **¿Hace sentido de negocio?** **No del todo.** Construir un Centro de Distribución multimillonario en medio del desierto/zona rural para atender apenas ~3% de solicitudes dispersas, mientras se deja a dos zonas de alta demanda (Ensenada y Valle de Guadalupe) compartiendo un solo Cedis a mitad de carretera, es una mala decisión financiera causada por no filtrar las anomalías antes.

2. **¿Por qué la matemática de este algoritmo provocó esto?**
   * **K-Means optimiza la media aritmética (centro de gravedad)** minimizando la función objetivo de Inercia o suma de errores cuadráticos dentro del clúster:
     $$\text{WCSS} = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$
   * Al elevar las distancias euclidianas **al cuadrado**, los valores atípicos o ruido (ranchos lejanos al sur hasta la latitud `30.56°`) penalizan desproporcionadamente la función de costo y ejercen un **fuerte "tirón gravitacional"** sobre los centroides.
   * Además, **K-Means es un algoritmo de partición exhaustiva**: está obligado matemáticamente a asignar **el 100% de los puntos** a algún clúster (no tiene concepto de ruido), por lo que sacrifica un centroide completo para cubrir la larga cola de puntos rurales al sur.

---

## Fase 3: Misión Operativa (Mapa de Tráfico y Zonas Calientes)
### Selección del Algoritmo: **DBSCAN (`DBSCAN`)**
* **¿Por qué se adecua a este problema?** Necesitamos aislar las **3 manchas urbanas de alta densidad** y **descartar matemáticamente las entregas rurales dispersas (ruido/anomalías)**. **DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*) agrupa puntos basándose en la densidad vecinal (radio `eps` y mínimo de vecinos `min_samples`), permitiendo identificar regiones densas de forma arbitraria y clasificar automáticamente los puntos aislados en carretera bajo la etiqueta **`Cluster -1` (Ruido)**.

In [ ]:
# Aplicamos DBSCAN sobre los datos escalados
dbscan = DBSCAN(eps=0.15, min_samples=20)
df['Cluster_DBSCAN'] = dbscan.fit_predict(X_scaled)

# Resumen de puntos por clúster
resumen_dbscan = df.groupby('Cluster_DBSCAN').agg(
    Solicitudes=('Latitud', 'count'),
    Latitud_Media=('Latitud', 'mean'),
    Longitud_Media=('Longitud', 'mean')
).reset_index()
display(resumen_dbscan)

# Separamos el ruido (Cluster -1) de las manchas urbanas (Clusters 0, 1, 2)
mascara_ruido = df['Cluster_DBSCAN'] == -1
df_ruido = df[mascara_ruido]
df_urbano = df[~mascara_ruido]

plt.figure(figsize=(9, 9))

# Pintamos primero el ruido en color gris
plt.scatter(
    df_ruido['Longitud'],
    df_ruido['Latitud'],
    c='lightgray',
    s=18,
    alpha=0.7,
    label=f'Ruido / Zonas Rurales (Cluster -1: {len(df_ruido)} pts)'
)

# Pintamos las 3 manchas urbanas con colores distintivos
colores_urbanos = {0: '#1f77b4', 1: '#2ca02c', 2: '#ff7f0e'}
nombres_ciudades = {
    0: 'Cluster 0: Tijuana',
    1: 'Cluster 1: Ensenada',
    2: 'Cluster 2: Valle de Guadalupe (Francisco Zarco)'
}

for cid in sorted(df_urbano['Cluster_DBSCAN'].unique()):
    sub = df_urbano[df_urbano['Cluster_DBSCAN'] == cid]
    plt.scatter(
        sub['Longitud'],
        sub['Latitud'],
        c=colores_urbanos.get(cid, 'purple'),
        s=22,
        alpha=0.8,
        label=f"{nombres_ciudades.get(cid, f'Cluster {cid}')} ({len(sub)} pts)"
    )

plt.title('Fase 3: Misión Operativa - Zonas Urbanas vs. Ruido Rural con DBSCAN', fontsize=13, fontweight='bold')
plt.xlabel('Longitud (Eje X)')
plt.ylabel('Latitud (Eje Y)')
plt.legend(loc='lower left')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

### Preguntas de Análisis — Fase 3

1. **¿Usaron el mismo algoritmo que en la misión anterior?**
   * **No.** En la Fase 2 utilizamos **K-Means** (basado en centroides/partición por distancia), mientras que en la Fase 3 utilizamos **DBSCAN** (basado en densidad espacial).

2. **¿De no ser así, por qué se comportaron diferente?**
   * **K-Means** agrupa basándose en la cercanía a $K$ centroides esféricos y **obliga a que todos los puntos pertenezcan a un grupo**, lo cual hace que los puntos rurales distorsionen los centros y se agrupen artificialmente.
   * **DBSCAN**, en cambio, evalúa la **densidad local**: exige que un punto núcleo (*core point*) tenga al menos `min_samples` puntos dentro de un radio `eps`. Las ciudades tienen alta concentración de solicitudes por kilómetro cuadrado y superan ese umbral formando clústeres bien definidos; en cambio, los ranchos aislados a lo largo de la carretera no alcanzan la densidad mínima vecinal y son descartados matemáticamente como **ruido (`Cluster -1`)**.

3. **¿A qué ciudades/zonas reales corresponden las 3 manchas urbanas que encontró su algoritmo?**
   Al inspeccionar las coordenadas GPS promedio de cada mancha urbana detectada por DBSCAN:
   * **Mancha Urbana 1 (`Cluster 0` — Lat: `32.515° N`, Lon: `-117.037° W`, ~1,582 solicitudes):** Corresponde a **Tijuana**.
   * **Mancha Urbana 2 (`Cluster 1` — Lat: `31.867° N`, Lon: `-116.595° W`, ~1,027 solicitudes):** Corresponde al puerto y ciudad de **Ensenada**.
   * **Mancha Urbana 3 (`Cluster 2` — Lat: `32.096° N`, Lon: `-116.530° W`, 300 solicitudes):** Corresponde a la zona de **Valle de Guadalupe (Francisco Zarco / Ruta del Vino)**, ubicada al noreste de Ensenada (nota: en el contexto regional de Baja California a veces también se asocia el tercer polo con el corredor turístico de **Valle de Guadalupe / Rosarito / Tecate**, pero geográficamente la coordenada exacta `32.096, -116.530` cae directamente sobre **Valle de Guadalupe / Francisco Zarco**).